# Day 1: Easy Agentic Orchestration with LangChain

## Goal of the Day
By the end of this session, you will:
- Understand how **LangChain abstracts away** the boilerplate of building agentic systems
- Use the **`@tool` decorator** to expose Python functions to LLMs
- Build a **ReAct agent** that shows transparent step-by-step reasoning
- Apply tools strategically in a **betting simulation** that requires combining data research with decision-making

## Notebook Structure
1. **Introduction**: Why LangChain? The problem with DIY agents
2. **Part 1 - MathsAgent**: Making LLMs do reliable calculations
3. **Part 2 - Galactic Betting**: Build strategic agents for the Intergalactic Football League

---
## Setup

First, let's clone the course repository from GitHub and install the required dependencies.

In [ ]:
# Clone the course repo from GitHub (public repo)
import os
from google.colab import userdata

REPO_URL = "https://github.com/MAS-AID-AI-Project/weekend4_agentic_public.git"

# Clone only if not already cloned
if not os.path.exists('/content/weekend4_agentic_public'):
    !git clone {REPO_URL} /content/weekend4_agentic_public
else:
    # Pull latest changes
    !cd /content/weekend4_agentic_public && git pull

# Navigate to the project directory
os.chdir('/content/weekend4_agentic_public/exercises/WE4_3_langchain')

print("Repository cloned and navigated to project directory!")

In [ ]:
!pip install -q -r requirements_day1.txt

In [ ]:
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_openai import ChatOpenAI

import json
from IPython.display import display, HTML, Markdown

print("All libraries loaded successfully!")

In [ ]:
# Setup OpenRouter API client
# IMPORTANT: Add your OpenRouter API key as a Colab secret named 'OPENROUTER_API_KEY'
# Go to the 🔑 icon in the left sidebar → Add a secret named OPENROUTER_API_KEY

API_KEY = userdata.get('OPENROUTER_API_KEY')

if not API_KEY:
    print("WARNING --- Please add your OpenRouter API key as a Colab secret named OPENROUTER_API_KEY!")
else:
    llm = ChatOpenAI(
        model="google/gemini-2.0-flash-001",  
        openai_api_key=API_KEY,
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0
    )
    print("OpenRouter client initialized!")

---
# Introduction: Why LangChain?

## Recap: The Agentic Loop

Remember from the previous session, an **Agentic AI** system usually follows this pattern:

![Agentic Loop](resources/basic_agentic_loop.jpeg)

This loop continues until the LLM decides the task is complete.

## The Pain Points of DIY Agents

If you were to build tools and execution logic **from scratch**, you'd face several challenges:

### 1. **Verbose Tool Schemas**
```python
# What you'd have to write manually everytime you want to expose a function as a tool:
tool_schema = {
    "name": "customer_lookup",
    "description": "Look up customer information...",
    "parameters": {
        "type": "object",
        "properties": {
            "customer_id": {"type": "integer", "description": "..."}
        }
    }
}
```

### 2. **Error Handling Boilerplate**
- What if the tool fails - i.e. returns an error?
- What if the LLM returns malformed JSON?

### 3. **Orchestration Logic**
- Managing the loop manually.
- Tracking conversation history - i.e. chosing what you feed again as the loop goes on, with which format.
- Deciding when to stop.

### 4. **No Standardization**
- If your colleague writes an agent, can you easily plug it into your system? 
- Can you swap GPT for Gemini without rewriting everything?

## Enter LangChain

**LangChain** solves these problems by providing:

| Feature | Benefit |
|---------|--------|
| **`@tool` decorator** | Turns Python functions into structured, model-readable interfaces (schema + description) |
| **Pre-built Agents** | Provide standard reasoning loops (e.g., ReAct) that manage tool selection, execution, and iteration - i.e. Production tested execution strategy prompts |
| **AgentExecutor** | Handles loops, errors, and history automatically - i.e. if tool fail, the error it gives will be fed to the agent as text |
| **Provider Agnostic** | Same code works with OpenAI, Gemini, Anthropic, etc. |
| **Community Ecosystem** | Thousands of pre-built tools, no need to re-invent the wheel everytime! |

Let's see this in action!

---
# Part 1: The MathsAgent - Making LLMs Do Reliable Math

## 1.1 The Problem: LLMs and Mathematics

It's well-known that LLMs can struggle with arithmetic. Let's see this in action.

In [ ]:
# Let's ask the LLM to do some multiplication WITHOUT tools
response = llm.invoke("What is 47 times 89? Just give me the number, no explanation.")
print(f"LLM's answer: {response.content}")
print(f"Actual answer: {47 * 89}")
print(f"Correct? {str(47 * 89) in response.content}")

In [ ]:
# Try a harder one
response = llm.invoke("What is 782379273 times 91566341? Just give me the number.")
print(f"LLM's answer: {response.content}")
print(f"Actual answer: {782379273 * 91566341}")
print(f"Correct? {str(782379273 * 91566341) in response.content}")

### The Business Case

Imagine you're building an AI assistant that helps verify expense reports from your accounting department. If the LLM makes arithmetic errors, you could:
- Approve incorrect expenses
- Flag legitimate expenses as fraudulent
- Lose trust in your AI system

**Solution**: Give the LLM a calculator tool!

---
## 1.2 Creating a Tool with LangChain

Watch how simple this is with LangChain's `@tool` decorator:

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(numbers: str) -> int:
    """Multiply two integers given as 'a, b'. ONLY HANDLES WHOLE INTEGERS, ROUND IF NEEDED"""
    
    # Converting a and b from string to int.
    # Note that we need this awkward translation, since the ReAct agent that we will use below only works with
    # single input tools - THERE ARE AGENTIC PLANS OUT THERE THAT HANDLE MULTI INPUTS OFF THE SHELF!
    a_str, b_str = numbers.split(",")
    a, b = int(a_str.strip()), int(b_str.strip())

    return a * b

# That's it! Let's inspect what LangChain created for us:
print("Tool Name:", multiply.name)
print("Tool Description:", multiply.description)
print("\nAuto-generated Schema:")
print(json.dumps(multiply.args_schema.model_json_schema(), indent=2))

### Compare with a Manual Approach

**LangChain (4 lines):**
```python
@tool
def multiply(numbers: str) -> int:
    """Multiply two integers given as 'a, b'."""
    
    a_str, b_str = numbers.split(",")
    a, b = int(a_str.strip()), int(b_str.strip())

    return a * b
```

**Manual (15+ lines):**
```python
multiply_schema = {
    "name": "multiply",
    "description": "Multiply two numbers together",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"}
        },
        "required": ["a", "b"]
    }
}

def multiply_executor(a, b):
    return a * b

# We would have to explicitly register the tool in a tool registry, to link schema and execution logic.
TOOL_REGISTRY["multiply"] = {"schema": multiply_schema, "executor": multiply_executor}
```

The `@tool` decorator automatically extracts:
- Function name -> tool name
- Docstring -> description
- Type hints -> parameter schema

**Beware** that this operator is **not all powerful**:
- If you had to define constraint logic, such as "a always positive", then you'd have to play with so call `Pydantic Schemas` - This is outside the scope of this session.

## 1.3 Creating the ReAct Agent

Now let's create an agent that can use our tool. We'll use the **ReAct** (Reasoning + Acting) pattern.

### What is ReAct?

ReAct agents follow a simple but powerful loop:

1. **Thought**: The LLM reasons about what to do next
2. **Action**: The LLM chooses a tool to use
3. **Action Input**: The LLM provides input for the tool
4. **Observation**: The tool result is returned to the LLM
5. **Repeat** until the LLM decides it has the final answer

This pattern makes the agent's reasoning **transparent** - you can see exactly how it thinks through a problem!

In [ ]:
# Step 1: Define our tools
tools = [multiply]

# Step 2: Load the official ReAct prompt from LangChain Hub
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/react")

print("Loaded ReAct prompt from LangChain Hub")
print("\nPrompt source: hwchase17/react")
print(f"Prompt type: {type(prompt).__name__}")
print("\nYou can view it at: https://smith.langchain.com/hub/hwchase17/react")

# Let's see what the prompt looks like:
print("\n" + "="*60)
print("PROMPT TEMPLATE:")
print("="*60)
print(prompt.template)

### A Note on Prompts and Parsers

You can swap this prompt with other ReAct-style prompts from [LangChain Hub](https://smith.langchain.com/hub/), such as `hwchase17/react-chat` or community contributions.

**However**, there's a catch: the prompt format is **tightly coupled** to the output parser. The ReAct prompt expects the LLM to output text like:
```
Thought: I need to multiply these numbers
Action: multiply
Action Input: 47, 89
```

The `create_react_agent` function uses a parser that looks for exactly this format. If you use a different prompt style (e.g., JSON-based), you'd need a different agent factory (e.g., `create_json_chat_agent`).

> **Note**: Modern LangChain (v1+) handles this differently with `create_agent()`, which abstracts away this coupling. We use the classic ReAct approach here because the explicit Thought/Action/Observation loop is clearer for learning how agents reason.

In [ ]:
# Step 3: Create the ReAct agent
agent = create_react_agent(llm, tools, prompt)

# Step 4: Wrap in AgentExecutor (handles the loop, errors, etc.)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # This shows us the Thought/Action/Observation cycle!
    handle_parsing_errors=True,
    max_iterations=5
)

print("MathsAgent created successfully!")

---
## 1.4 Running the Agent

Let's test our MathsAgent! Watch the verbose output to see the ReAct reasoning.

In [ ]:
# Test 1: Simple multiplication
result = agent_executor.invoke({"input": "What is 47 times 89?"})
print("\n" + "="*50)
print(f"Final Answer: {result['output']}")
print(f"Expected: {47 * 89}")

In [ ]:
# Test 2: Harder multiplication
result = agent_executor.invoke({"input": "What is 782379273 times 91566341?"})
print("\n" + "="*50)
print(f"Final Answer: {result['output']}")
print(f"Expected: {782379273 * 91566341}")

### Understanding the Output

Notice how the agent:
1. **Thought**: "I need to multiply these numbers"
2. **Action**: `multiply` with `a=47, b=89`
3. **Observation**: `4183`
4. **Thought**: "I now know the answer"
5. **Final Answer**: `4183`

This is the power of ReAct - transparent, step-by-step reasoning!

---
## 1.5 Exercise: The Discount Calculator 🎯

### Scenario

You work at a retail company. The finance team needs to calculate discounted prices:

> "A product costs **$247.50**. We're offering a **15% discount**. What's the final price?"

The calculation requires:
1. Calculate the discount amount: `247.50 × 0.15 = 37.125`
2. Subtract from original: `247.50 - 37.125 = 210.375`

### Your Task

Our current agent only has a `multiply` tool. To solve this problem, you need to add **one more tool**.

**Choose your approach:**

| Option A: Add Tool | Option B: Discount Tool |
|-------------------|-------------------|
| `add(a, b)` - Simple addition | `apply_discount(price, discount_percent)` - Does both steps |
| Agent must do 2 operations | Agent does 1 operation |
| More flexible | More specialized |

In [ ]:
# Option A Add Tool cell
# TODO: Complete the tool below with the correct add behaviour
@tool
def add(numbers: str) -> float:
    """Add two floats given as 'a, b'."""
    
    a_str, b_str = numbers.split(",")
    a, b = float(a_str.strip()), float(b_str.strip())

    return ???

In [ ]:
# Option B Discount Tool cell
# Complete the tool below with the correct discount behaviour
@tool
def apply_discount(numbers: str) -> float:
    """Given 'price, discount_percent' in numbers, apply discount_percent to price and return the discounted price."""
    
    price_str, discount_percent_str = numbers.split(",")
    price, discount_percent = float(price_str.strip()), float(discount_percent_str.strip())
    
    return ???

In [ ]:
# TODO: Create the updated agent with both tools
# Hint: tools = [multiply, your_new_tool]

# YOUR CODE HERE:
discount_tools = [multiply]  # TODO: Add your tool here!

discount_agent = create_react_agent(llm, discount_tools, prompt)
discount_executor = AgentExecutor(
    agent=discount_agent,
    tools=discount_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)

print("Discount Agent created!")

In [ ]:
# Test your discount agent!
result = discount_executor.invoke({
    "input": "A product costs $247.50 and we're offering a 15% discount. What's the final price?"
})

print("\n" + "="*50)
print(f"Agent's Answer: {result['output']}")
print(f"Expected: ${247.50 * (1 - 0.15)}")

**Have you noticed?** If you went for the `add` and `multiply`option, then you lost precision during the `multiply`tool call, since it is designed to handle integers and not floating points!

---
## 1.6 The Power of Community: Pre-built Tools

One of LangChain's biggest strengths is its **ecosystem of pre-built tools**. Instead of writing everything from scratch, you can import ready-made tools for common tasks.

### Available Tool Categories

| Category | Examples |
|----------|----------|
| **Search** | DuckDuckGo, Tavily, Google Search |
| **Knowledge** | Wikipedia, Arxiv, PubMed |
| **Code** | Python REPL, Shell |
| **APIs** | OpenWeatherMap, Wolfram Alpha |

Unfortunately there is no proper centralized exhaustive list of tools, but a good overview can be found at [Here](https://docs.langchain.com/oss/python/integrations/tools)!

Let's see how easy it is to add web search and Wikipedia to an agent!

In [ ]:
# Import pre-built tools from langchain-community
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Create the tools - no API keys needed for these!
search_tool = DuckDuckGoSearchRun(
    name="web_search",
    description="Search the web for current information. Use this for recent events or facts you're unsure about."
)

wiki_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wiki_tool = WikipediaQueryRun(
    api_wrapper=wiki_wrapper,
    name="wikipedia", 
    description="Look up factual information on Wikipedia. Good for historical facts, biographies, and general knowledge."
)

research_tools = [search_tool, wiki_tool]

print("Pre-built tools loaded!")
print("\nAvailable tools:")
for t in research_tools:
    print(f"  - {t.name}: {t.description}")

In [ ]:
# Quick demo: Create a research agent with these tools
research_agent = create_react_agent(llm, research_tools, prompt)
research_executor = AgentExecutor(
    agent=research_agent,
    tools=research_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

In [ ]:
# Try it out for a Web Search! (Hopefully it should understand that it needs to use the DuckDuckGo Search).
result = research_executor.invoke({
    "input": "Who won the 2022 FIFA World Cup and what was the final score?"
})

print("\n" + "="*50)
print(f"Answer: {result['output']}")

In [ ]:
# Try it out for a Wikipedia search!
result = research_executor.invoke({
    "input": "Explain theory of relativity to me, using the Wikipedia's page content"
})

print("\n" + "="*50)
print(f"Answer: {result['output']}")

---
# Part 2: Intergalactic Football Betting Agents

Now that we've mastered the basics, let's apply these skills to a more complex scenario!

---
## 2.1 Starting the Galactic Frontends

We've built two web interfaces for you to explore:

1. **GalacticBets.io** - The official betting platform
   - View tournament standings
   - See completed match results
   - Place bets on upcoming matches

2. **StarScoop** - The galaxy's premier paparazzi news site
   - Read gossip about teams and players
   - Search for insider information
   - Some news might hint at match outcomes...

Let's launch them!

In [ ]:
# Import the galactic betting module
import galactic_betting
from google.colab import output
from IPython.display import display, HTML

print("Module loaded and ready!")

In [ ]:
# Start Betting Platform
betting_port = galactic_betting.start_betting_server()
display(HTML('<h3 style="color:#00ffff;">🎰 Betting Platform</h3>'))
output.serve_kernel_port_as_iframe(betting_port, height=800)

In [ ]:
# Start News Site
news_port = galactic_betting.start_news_server()
display(HTML('<h3 style="color:#ff00ff;">📰 StarScoop News</h3>'))
output.serve_kernel_port_as_iframe(news_port, height=800)

### 🎯 Exploration Task (5 minutes)

Before building your agent, explore the frontends:

1. **GalacticBets.io**:
   - Which team finished first in the regular season?
   - What matches are available for betting?
   - What are the odds for the Upper Bracket Final?

2. **StarScoop**:
   - Can you spot which news might be useful vs. just gossip?

---
## 2.2 The Betting Tools

Your agent will have access to 4 tools that interact with the betting system:

In [ ]:
# Get the pre-built betting tools
betting_tools = galactic_betting.get_all_tools()

# Let's see what tools we have:
print("Available Betting Tools:")
print("=" * 60)
for tool in betting_tools:
    print(f"\n📌 {tool.name}")
    print(f"   {tool.description}")

In [ ]:
# Let's test each tool manually to understand what they return

# Tool 1: Search paparazzi news
print("\n" + "="*60)
print("TESTING: search_paparazzi_news")
print("="*60)
news_results = galactic_betting.search_paparazzi_news.invoke("Meteor Mavericks")
print(f"Found {len(news_results)} articles about Meteor Mavericks")
if news_results:
    print(f"Latest headline: {news_results[0]['headline'][:80]}...")

In [ ]:
# Tool 2: Get match history
print("\n" + "="*60)
print("TESTING: get_match_history")
print("="*60)
history = galactic_betting.get_match_history.invoke("Stellar Strikers")
print(f"Team: {history['team']} {history['logo']}")
print(f"Record: {history['summary']['wins']}W - {history['summary']['draws']}D - {history['summary']['losses']}L")
print(f"Goals: {history['summary']['goals_for']} scored, {history['summary']['goals_against']} conceded")

In [ ]:
# Tool 3: Get betting odds
print("\n" + "="*60)
print("TESTING: get_betting_odds")
print("="*60)
odds_info = galactic_betting.get_betting_odds.invoke({})
print(f"Your balance: {odds_info['current_balance']} GC")
print(f"\nUpcoming matches:")
for match in odds_info['upcoming_matches']:
    print(f"  [{match['match_id']}] {match['home']} vs {match['away']}")
    print(f"       Odds: {match['home']}: {match['odds'][match['home']]} | {match['away']}: {match['odds'][match['away']]}")

In [ ]:
# Tool 4: Place a bet (let's try a small one)
print("\n" + "="*60)
print("TESTING: place_bet")
print("="*60)
# Uncomment to test (this will actually place a bet!):
# bet_result = galactic_betting.place_bet.invoke({"match_id": 201, "team_to_win": "Stellar Strikers", "amount": 10})
# print(json.dumps(bet_result, indent=2))
print("(Skipping actual bet placement for now - you'll do this with your agent!)")

---
## 2.3 Building Your Betting Agent

Now it's time to create your betting agent! The key is crafting a good **betting strategy**.

### Strategy Considerations

Think about:
- Should your agent bet on favorites (lower odds, safer) or underdogs (higher odds, riskier)?
- How much research should it do before betting?
- Should it spread bets across multiple matches or go all-in on one?
- How should it weigh statistics vs. gossip/news?

In [ ]:
# Reset wallet to start fresh
galactic_betting.reset_wallet()
print("Wallet reset to 1000 GC!")

In [ ]:
# Create the betting agent using ReAct
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/react")

betting_agent = create_react_agent(llm, betting_tools, prompt)
betting_executor = AgentExecutor(
    agent=betting_agent,
    tools=betting_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=20  # Allow more iterations for research, you can push it even more depending on your strategy!
)

print("Betting Agent created!")

In [ ]:
# Set your strategy there!
strategy = """
Analyze the upcoming matches and place strategic bets.

First, ...
Then, ...
"""

# Run the betting agent!
result = betting_executor.invoke({
    "input": strategy
})

print("\n" + "="*60)
print("AGENT'S FINAL SUMMARY:")
print("="*60)
print(result['output'])

In [ ]:
# Check what bets were placed
placed_bets = galactic_betting.get_placed_bets()
print("\nYOUR PLACED BETS:")
print("="*60)
for bet in placed_bets:
    print(f"\nMatch: {bet['match_description']}")
    print(f"  Betting on: {bet['team_to_win']} @ {bet['odds']}x")
    print(f"  Amount: {bet['amount']} GC")
    print(f"  Potential winnings: {bet['potential_winnings']} GC")

print(f"\nRemaining Balance: {galactic_betting.get_wallet_balance()} GC")

---
## 2.5 Reveal the Results! 🎰

The moment of truth! Let's see how your agent's bets performed.

**Note**: You can also click "REVEAL RESULTS" on the GalacticBets.io website for a fancier display!

In [ ]:
# REVEAL THE RESULTS!
# Only run this when you're ready - it can't be undone!

results = galactic_betting.reveal_results()

print("\n" + "🏆"*20)
print("\n          MATCH RESULTS REVEALED!")
print("\n" + "🏆"*20)

print("\n📊 ACTUAL MATCH OUTCOMES:")
print("-"*40)
for match in results['match_results']:
    print(f"\n{match['round']}:")
    print(f"  {match['match']}")
    print(f"  Score: {match['score']}")
    print(f"  Winner: ⭐ {match['winner']}")

print("\n" + "="*60)
print("💰 YOUR BET RESULTS:")
print("="*60)
for bet in results['bets']:
    emoji = "✅" if bet['status'] == 'won' else "❌"
    print(f"\n{emoji} {bet['match_description']}")
    print(f"   You bet on: {bet['team_to_win']}")
    print(f"   Result: {bet['result']}")

print("\n" + "="*60)
starting_balance = 1000
profit = results['final_balance'] - starting_balance
profit_emoji = "📈" if profit >= 0 else "📉"

print(f"\n{profit_emoji} FINAL RESULTS:")
print(f"   Starting Balance: 1000 GC")
print(f"   Final Balance: {results['final_balance']:.2f} GC")
print(f"   Net Profit/Loss: {'+' if profit >= 0 else ''}{profit:.2f} GC")
print("\n" + "🏆"*20)

---
## 2.6 Discussion & Reflection

### Questions to Consider

1. **Did your agent find the "hidden signals" in the news?**
   - The coach/captain reconciliation for Meteor Mavericks?
   - The goalkeeper injury for Andromeda Asteroids?
   - The training record for Stellar Strikers?

2. **Did your agent distinguish signal from noise?**
   - Sad poetry from a backup player?
   - Mascot winning a fashion award?
   - Chef's recipe?

3. **How did prompt engineering affect results?**
   - Try different strategies and compare!
   - What if you told it to bet only on underdogs?
   - What if you told it to ignore statistics and focus on gossip?

### Key Takeaways

- **Agents are only as good as their tools** - Give them access to the right information
- **Prompt engineering matters** - Strategy definition significantly impacts behavior
- **Signal vs. Noise** - Agents need guidance on what information is relevant
- **Transparency** - ReAct's verbose mode helps debug and understand agent reasoning

---
## Bonus: Try Different Strategies!

Reset your wallet and try a different approach. See how results change!

In [ ]:
# Reset and try again with a different strategy!
galactic_betting.reset_wallet()

# Then re-run starting from where you defined your strategy!

---
## Cleanup

Stop the frontend servers when you're done.

In [ ]:
# Stop the servers
galactic_betting.stop_servers()
print("Servers stopped. Thanks for playing!")

---
# Summary

## What We Built

### Part 1: MathsAgent
- Demonstrated that LLMs can fail at arithmetic
- Created a `multiply` tool using the `@tool` decorator
- Built a ReAct agent that uses the tool to get correct results
- Extended with additional tools (add/discount) for multi-step calculations

### Part 2: Betting Agents
- Used 4 tools: news search, match history, betting odds, and placing bets
- Crafted a strategy prompt to guide agent behavior
- Observed how agents filter signal from noise in paparazzi articles

## Key Concepts

| Concept | What It Does |
|---------|--------------|
| `@tool` decorator | Turns a Python function into a tool with auto-generated schema |
| ReAct pattern | Thought -> Action -> Observation loop for transparent reasoning |
| `AgentExecutor` | Manages the loop, handles errors, tracks history |
| Prompt engineering | Shapes how agents prioritize and use information |

## Main Takeaways

1. **LangChain handles the plumbing** - no manual schema writing or loop management
2. **Tools extend what LLMs can do** - calculations, web search, API calls
3. **ReAct makes reasoning visible** - easier to debug and understand agent decisions
4. **Strategy prompts matter** - the same tools can produce different outcomes based on instructions